In [1]:
from playpen import EpisodeBuffer
%load_ext autoreload
%autoreload 2

This notebook demonstrates how to train a language model to play **Wordle** using
**Group Relative Policy Optimization (GRPO)** with TRL and the clemcore OpenEnv integration.

You will learn how to:

1. Set up the environment and install dependencies.
2. Connect to a Wordle game server using OpenEnv.
3. Implement a GRPO-compatible agent that tracks episode trajectories.
4. Run GRPO training with TRL using the OpenEnv game as the environment.

**Key concepts:**
- **GRPO**: A reinforcement learning algorithm that compares K completions from the same prompt to compute advantages, without requiring a separate reference model.
- **OpenEnv**: A Gymnasium-style API for interacting with text-based environments.
- **rollout_func**: A custom function that lets TRL collect trajectories from an environment instead of using standard text generation.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/phisad/playpen/blob/main/examples/openenv/wordle-trl.ipynb
)

# 1. Environment setup and dependency installation

## 1.1. Setup environment

We start by specifying:

- The game name (here we use `"wordle"`, but this pattern works for any 1‑player game).
- `CLEMBENCH_HOME`, the local path where the `clembench` repository is located.
  This environment variable is used by the CLI and Python APIs to locate games.

In [2]:
import os

# Specify the game name here (this code can be adapted to any 1-player game)
GAME_NAME = "wordle"

# Local clone location of the clembench repository
CLEMBENCH_HOME = os.path.expanduser("~/git/clembench")

# Expose CLEMBENCH_HOME so the clem framework can find the games
os.environ["CLEMBENCH_HOME"] = CLEMBENCH_HOME

## 1.2. Install games and dependencies

In this step we:

1. Clone the `clembench` repository (skip if you already have it).
2. Install its Python dependencies into the current Jupyter kernel.
3. Verify that `clembench` is installed and that the game is visible.

You should:

- See a valid version printed by `clem --version`.
- See `wordle` listed by `clem list games -s wordle`.

If you do **not** see `wordle`, double‑check `CLEMBENCH_HOME` and that the clone succeeded.

In [ ]:
# Clone the clembench repo (safe to re-run; git will warn if it already exists)
!git clone https://github.com/clp-research/clembench $CLEMBENCH_HOME

# Install the requirements into the Python kernel
%pip install -r $CLEMBENCH_HOME/requirements.txt

# Make tqdm usable in Jupyter notebooks
%pip install --upgrade ipywidgets jupyter_client

In [ ]:
# Sanity check: version + confirm that the game is an available game
!clem --version
!clem list games -s $GAME_NAME

# 2. Connect to the Wordle game server

The Wordle game runs as an OpenEnv server that our agent will interact with.
Each call to `env.reset()` starts a new game with a random target word, and
`env.step(action)` submits a guess and returns feedback.

## 2.1. Starting the server and connecting the client

Open a terminal in the notebook folder and run the `clem serve` command to start the OpenEnv environment:

```bash
clem serve --game wordle
```

The output should look similar to:
```
INFO:     Started server process [73210]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
```

From this log you obtain the host and port (here `http://0.0.0.0:8000`) that the client should connect to.

In [3]:
from clemcore.clemgame import ClemGameEnv

game_env = ClemGameEnv(base_url="http://0.0.0.0:8000")

# 3. Implement the GRPO-compatible Wordle agent

To train with GRPO using an interactive environment, we need to:

1. **Track the full episode trajectory** as a single prompt-completion pair
2. **Distinguish model tokens from environment tokens** using an `env_mask`
3. **Capture log probabilities** for each generated token

The trajectory structure follows TRL's pattern:
- `prompt_ids`: The initial game prompt (set once at episode start)
- `completion_ids`: All model outputs + environment feedback concatenated across turns
- `logprobs`: Log probability for each token (0.0 for environment tokens)
- `env_mask`: 1 for model-generated tokens, 0 for environment tokens

GRPO only computes gradients on tokens where `env_mask=1`, but uses the full
trajectory to calculate rewards and advantages.

## 3.1. Define data containers for episode trajectories

We define two dataclasses:
- `GrpoEpisodeRollout`: Holds a single episode's trajectory data
- `GrpoEpisodeRollouts`: Aggregates multiple episodes for a training batch

These will be converted to a dictionary via `asdict()` and returned to TRL's
`GRPOTrainer`, which expects keys: `prompt_ids`, `completion_ids`, `logprobs`, and `env_mask`.

In [4]:
from dataclasses import dataclass, field
from playpen.agents.openenv import ClemGameEnvAgent


@dataclass
class GrpoEpisodeRollout:
    """Collect training info about a single episode.
    
    Following TRL's pattern, we accumulate all turns into a single completion
    sequence, using env_masks to distinguish model tokens (1) from env tokens (0).
    """
    prompt_ids: list[int] = field(default_factory=list)
    completion_ids: list[int] = field(default_factory=list)
    logprobs: list[float] = field(default_factory=list)
    env_masks: list[int] = field(default_factory=list)

    def reset(self):
        self.prompt_ids.clear()
        self.completion_ids.clear()
        self.logprobs.clear()
        self.env_masks.clear()


@dataclass
class GrpoEpisodeRollouts:
    """Collect training info about all episodes for a batch."""
    prompt_ids: list[list[int]] = field(default_factory=list)
    completion_ids: list[list[int]] = field(default_factory=list)
    logprobs: list[list[float]] = field(default_factory=list)
    env_mask: list[list[int]] = field(default_factory=list)

    def append(self, rollout: GrpoEpisodeRollout):
        """Append a completed episode rollout."""
        self.prompt_ids.append(list(rollout.prompt_ids))
        self.completion_ids.append(list(rollout.completion_ids))
        self.logprobs.append(list(rollout.logprobs))
        self.env_mask.append(list(rollout.env_masks))

    def reset(self):
        self.prompt_ids.clear()
        self.completion_ids.clear()
        self.logprobs.clear()
        self.env_mask.clear()


.--------------..--------------..--------------..--------------..--------------..--------------..--------------.
|   ______     ||   _____      ||      __      ||  ____  ____  ||   ______     ||  _________   || ____  _____  |
|  |_   __ \   ||  |_   _|     ||     /  \     || |_  _||_  _| ||  |_   __ \   || |_   ___  |  |||_   \|_   _| |
|    | |__) |  ||    | |       ||    / /\ \    ||   \ \  / /   ||    | |__) |  ||   | |_  \_|  ||  |   \ | |   |
|    |  ___/   ||    | |   _   ||   / ____ \   ||    \ \/ /    ||    |  ___/   ||   |  _|  _   ||  | |\ \| |   |
|   _| |_      ||   _| |__/ |  || _/ /    \ \_ ||    _|  |_    ||   _| |_      ||  _| |___/ |  || _| |_\   |_  |
|  |_____|     ||  |________|  |||____|  |____|||   |______|   ||  |_____|     || |_________|  |||_____|\____| |
'--------------''--------------''--------------''--------------''--------------''--------------''--------------'



## 3.2. Implement the WordleAgent

The `WordleAgent` extends `ClemAgent` and handles:

1. **Generation**: Uses TRL's `generate_rollout_completions()` with vLLM for fast inference
2. **Trajectory tracking**: Accumulates tokens and logprobs across all turns

Key methods:
- `track_env_completion()`: On first turn, captures the initial prompt. On subsequent turns,
  tokenizes the environment feedback and adds it with `env_mask=0`.
- `track_agent_completion()`: Adds model-generated tokens with `env_mask=1`.
- `act()`: Orchestrates tracking → generation → tracking → return response.

The agent uses `self.history` (inherited from `ClemAgent`) to build the full
conversation context for each generation call.

In [5]:
import trl
from playpen.agents import ClemObservation, ClemAgent
from trl.experimental.openenv import generate_rollout_completions


class WordleAgent(ClemAgent):
    """Agent that plays Wordle using TRL's GRPOTrainer for generation.
    
    Handles all tokenization and env_mask logic internally, accumulating
    the full episode trajectory for GRPO training.

    This implementation is based on the wordle openenv example given in the trl repository at
    https://github.com/huggingface/trl/blob/v0.28.0/examples/scripts/openenv/wordle.py
    """

    def __init__(self, trainer: trl.GRPOTrainer):
        super().__init__()
        self.trainer = trainer
        self.tokenizer = trainer.processing_class
        self.episode = GrpoEpisodeRollout()
        self._first_turn = True

    def track_env_completion(self, last: ClemObservation):
        if self._first_turn:  # On the first turn, set prompt_ids from the initial observation
            prompt_text = self.tokenizer.apply_chat_template(self.history, add_generation_prompt=True, tokenize=False)
            self.episode.prompt_ids = self.tokenizer.encode(prompt_text, add_special_tokens=False)
            self._first_turn = False
            return
        # Not first turn: the observation content is env feedback from the previous action
        env_feedback_ids = self.tokenizer.encode(last.content, add_special_tokens=False)
        self.episode.completion_ids.extend(env_feedback_ids)
        self.episode.logprobs.extend([0.0] * len(env_feedback_ids))
        self.episode.env_masks.extend([0] * len(env_feedback_ids))

    def track_agent_completion(self, outputs):
        self.episode.completion_ids.extend(outputs["completion_ids"])
        self.episode.logprobs.extend(outputs["logprobs"])
        self.episode.env_masks.extend([1] * len(outputs["completion_ids"]))

    def act(self, last: ClemObservation) -> str:
        self.track_env_completion(last)
        # We use text here so that we can better track the agent tokens versus the env tokens
        prompt_text = self.tokenizer.apply_chat_template(self.history, add_generation_prompt=True, tokenize=False)
        outputs = generate_rollout_completions(self.trainer, [prompt_text])[0]
        self.track_agent_completion(outputs)
        response = outputs.get("text") or self.tokenizer.decode(outputs["completion_ids"], skip_special_tokens=True)
        return response

    def get_episode(self) -> "GrpoEpisodeRollout":
        """Return the accumulated episode rollout for GRPO training."""
        return self.episode

    def reset(self):
        """Reset agent state for a new episode."""
        super().reset()
        self.episode.reset()
        self._first_turn = True

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
/var/folders/qn/h5lsrtbs0rn06p25spz4qvbm0000gn/T/ipykernel_93917/1938421726.py:3: TRLExperimentalWarning: You are importing from 'trl.experimental'. APIs here are unstable and may change or be removed without notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  from trl.experimental.openenv import generate_rollout_completions
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


NameError: name 'LRScheduler' is not defined

## 3.3. Define the rollout function for TRL

TRL's `GRPOTrainer` supports a `rollout_func` parameter that overrides the default
text generation loop. This lets us interact with the OpenEnv game instead.

**How it works:**
1. TRL calls `rollout_func(prompts, trainer)` with prompts repeated K times (K = `num_generations`)
2. We run K episodes sequentially (one per prompt), collecting trajectories
3. We return a dict with `prompt_ids`, `completion_ids`, `logprobs`, and `env_mask`
4. TRL uses this data for advantage computation and policy gradient updates

Note: Since we only have one game server, episodes run sequentially. For better
throughput, you could spawn multiple game servers and parallelize.

In [ ]:
from dataclasses import asdict


def rollout_episode(env: ClemGameEnv, agent: ClemGameEnvAgent) -> GrpoEpisodeRollout:
    """Play a single episode of Wordle and collect GRPO training data.
    
    The agent handles all tokenization and env_mask logic internally.
    
    Returns:
        GrpoEpisodeRollout with accumulated prompt_ids, completion_ids, logprobs, and env_masks.
    """
    obs = env.reset()
    while not obs.done:
        action = agent(obs)
        obs = env.step(action)
    return agent.wrapped_agent.get_episode()


def rollout_func(prompts: list[str], trainer: trl.GRPOTrainer) -> dict:
    """Custom rollout function for TRL GRPOTrainer with OpenEnv.
    
    Note:
        rollout_func receives prompts already duplicated K times (num_generations).
        For example, with batch size 4 and num_generations 8, there are 32 prompts.
        Since Wordle always starts with the same initial state, we ignore the prompt
        content and just run that many episodes.
    """
    # Create the TRL wordle agent and apply the openenv wrapper for convenience
    agent = ClemGameEnvAgent(WordleAgent(trainer))
    rollouts = GrpoEpisodeRollouts()
    try:
        # We only have a single env started, so we must go sequentially here
        for _ in prompts:
            rollout = rollout_episode(game_env, agent)
            rollouts.append(rollout)
            agent.reset()  # Reset for next episode
        return asdict(rollouts)
    finally:
        rollouts.reset()
        agent.reset()

# 4. Configure and run GRPO training

Now we set up the `GRPOTrainer` with:
- **vLLM colocate mode**: Fast inference on the same GPU as training
- **LoRA**: Only trains adapter weights, not the full model
- **8-bit quantization** (optional): Further reduces memory for smaller GPUs

**Memory estimates for Llama 3 8B + LoRA:**

| Configuration | GPU Memory |
|---------------|------------|
| bf16 (no quantization) | ~45 GB |
| 8-bit quantization | ~28 GB |

On an A100 80GB, you can skip quantization. Use 8-bit for 40GB A100 or 24GB consumer GPUs.

## 4.1. Load the training dataset

We load game instances from the playpen-data dataset. Each instance represents
a game configuration (e.g., a specific target word). The dataset is filtered
to only include Wordle instances.

Note: In this setup, the dataset mainly controls how many training steps we run.
The actual game content comes from the OpenEnv server, which generates new games
on each `env.reset()` call.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("colab-potsdam/playpen-data", "instances", split="train")
dataset = dataset.filter(lambda game_instance: game_instance["game"] == "wordle")

## 4.2. Configure and start training

Key configuration options:

**GRPOConfig:**
- `use_vllm=True, vllm_mode="colocate"`: Use vLLM for generation on the same GPU
- `num_generations=4`: Compare 4 completions per prompt for advantage estimation
- `disable_dropout=True`: Ensures consistent policy during generation and training
- `max_completion_length=2048`: Must fit the full episode (6 turns × explanation + guess)

**LoraConfig:**
- `r=16, lora_alpha=32`: LoRA rank and scaling factor
- `target_modules="all-linear"`: Apply LoRA to all linear layers

**BitsAndBytesConfig (optional):**
- `load_in_8bit=True`: Quantize base model to reduce memory
- Set to `None` on 80GB+ GPUs where memory is not a constraint

The training loop will:
1. Sample a batch of prompts from the dataset
2. Call `rollout_func` to play episodes and collect trajectories
3. Compute advantages by comparing rewards across the K completions
4. Update LoRA weights via policy gradient

In [ ]:
# TRL supports rollout_func only for vllm mode
# TRL has problems with PEFT + vllm in colocate mode, but requires 2 GPUs to run on server mode: one for inference and one for training (which makes training slower; and misses kind of the point of GRPO not having a separate model loaded)
# hence, TRL full training with colocate mode and
from peft import LoraConfig

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
grpo_config = trl.GRPOConfig(
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.6,
    num_generations=4,  # Default
    per_device_train_batch_size=4,  # Default
    num_train_epochs=10,
    disable_dropout=True,
    max_prompt_length=None,
    max_completion_length=2048,  # Should capture full episode; note that the model is asked to give an explanation
    output_dir=f"models/grpo/wordle/{MODEL_ID}"
)
peft_config = LoraConfig(  # see https://huggingface.co/docs/trl/sft_trainer#training-adapters
    r=16, lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    modules_to_save=["lm_head", "embed_token"],
    task_type="CAUSAL_LM",
)

# Optional: 8-bit quantization to reduce memory (~28GB vs ~45GB for bf16)
bnb_config = None  # Default: Set to None on 80GB+ GPUs

# Uncomment for 40GB A100 or consumer GPUs
# from transformers import BitsAndBytesConfig
# bnb_config = BitsAndBytesConfig(load_in_8bit=True)

grpo_trainer = trl.GRPOTrainer(
    model=MODEL_ID,
    rollout_func=rollout_func,  # repeats each game instance in batch K times (using RepeatSampler)
    train_dataset=dataset,
    args=grpo_config,
    peft_config=peft_config,
    model_init_kwargs={"quantization_config": bnb_config} if bnb_config else {}
)

# Train on the dataset; this will save only the adapters to the checkpoints directory
grpo_trainer.train()